# API Testing

Tests for the Literature AI core microservice API.

Before running, start the API with:
```bash
python -m literature_ai.core.api
```

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

import requests
import pandas as pd
from IPython.display import display, Markdown

BASE_URL = "http://localhost:8000"

## GET /embedding-models

Returns all available embedding runs. Use a `run_id` from this response to call `/search`.

In [ ]:
response = requests.get(f"{BASE_URL}/embedding-models")
response.raise_for_status()

payload = response.json()
runs = payload["runs"]
print(f"Found {len(runs)} embedding run(s)")

df_runs = pd.DataFrame(runs)
display(df_runs)

run_id = runs[0]["run_id"]
print(f"\nUsing run_id={run_id} for search")

## POST /search

Perform a vector similarity search using the selected `run_id`.

In [ ]:
QUERY = "deep learning optical coherence tomography retinal segmentation"
N_RESULTS = 5

response = requests.post(
    f"{BASE_URL}/search",
    json={"query": QUERY, "run_id": run_id, "n_results": N_RESULTS},
)
response.raise_for_status()

payload = response.json()
results = payload["results"]
print(f"Query   : {payload['query']!r}")
print(f"Run ID  : {payload['run_id']}")
print(f"Results : {len(results)}")

## Results

In [ ]:
for i, r in enumerate(results, 1):
    abstract_preview = (r["abstract"] or "")[:300]
    display(Markdown(
        f"### {i}. {r['title']} ({r['year']})\n\n"
        f"**Venue:** {r['venue']}  \n"
        f"**Citations:** {r['citationCount']}  \n"
        f"**Distance:** {r['distance']:.4f}  \n\n"
        f"> {abstract_preview}..."
    ))